# 🔍 Verify Saved Thin Cloud Model

This notebook verifies that the trained RL model is saved on Google Drive and can be loaded successfully.

In [ ]:
# Mount Google Drive
from google.colab import drive
import os

# Check if already mounted
if os.path.exists('/content/drive/MyDrive'):
    print('✅ Google Drive already mounted!')
else:
    drive.mount('/content/drive')
    print('✅ Google Drive mounted successfully!')

In [ ]:
# Scan entire Colab_Data folder
import os

colab_data_path = '/content/drive/MyDrive/Colab_Data'

print(f"📂 Scanning: {colab_data_path}\n")
print("=" * 60)

if os.path.exists(colab_data_path):
    for item in sorted(os.listdir(colab_data_path)):
        item_path = os.path.join(colab_data_path, item)
        if os.path.isdir(item_path):
            # Count files in directory
            try:
                files = os.listdir(item_path)
                file_count = len(files)
                # Calculate total size
                total_size = 0
                for f in files:
                    fp = os.path.join(item_path, f)
                    if os.path.isfile(fp):
                        total_size += os.path.getsize(fp)
                size_mb = total_size / (1024 * 1024)
                print(f"📁 {item}/ ({file_count} files, {size_mb:.2f} MB)")
                # Show files inside
                for f in sorted(files)[:10]:  # Show first 10 files
                    fp = os.path.join(item_path, f)
                    if os.path.isfile(fp):
                        fsize = os.path.getsize(fp) / (1024 * 1024)
                        print(f"   └── {f} ({fsize:.2f} MB)")
                    else:
                        print(f"   └── {f}/")
                if len(files) > 10:
                    print(f"   └── ... and {len(files) - 10} more")
            except Exception as e:
                print(f"📁 {item}/ (error reading: {e})")
        else:
            size_mb = os.path.getsize(item_path) / (1024 * 1024)
            print(f"📄 {item} ({size_mb:.2f} MB)")
    print("=" * 60)
else:
    print(f"❌ Path not found: {colab_data_path}")
    print("   Make sure Google Drive is mounted first!")

In [ ]:
# Check for saved models
import os
from pathlib import Path

print("🔍 Searching for saved thin cloud models...\n")

# Possible model locations
model_paths = [
    '/content/drive/MyDrive/Colab_Data/thin_cloud_v2',
    '/content/drive/MyDrive/Colab_Data/thin_cloud_multiobj',
    '/content/drive/MyDrive/Colab_Data/thin_cloud_multiobj_final',
]

found_models = []

for path in model_paths:
    if os.path.exists(path):
        files = os.listdir(path)
        zip_files = [f for f in files if f.endswith('.zip')]
        if zip_files:
            print(f"✅ Found models in: {path}")
            for f in sorted(zip_files):
                full_path = os.path.join(path, f)
                size_mb = os.path.getsize(full_path) / (1024 * 1024)
                print(f"   📦 {f} ({size_mb:.2f} MB)")
                found_models.append(full_path)
        else:
            print(f"⚠️  Folder exists but empty: {path}")
    else:
        print(f"❌ Not found: {path}")

print(f"\n📊 Total models found: {len(found_models)}")

if found_models:
    # Use the most recent/largest model
    best_model = max(found_models, key=lambda x: os.path.getsize(x))
    print(f"\n🎯 Best model to use: {best_model}")

In [ ]:
# Install dependencies
!pip install -q stable-baselines3 gymnasium

In [ ]:
# Try to load the model
from stable_baselines3 import PPO
import warnings
warnings.filterwarnings('ignore')

# Set the model path (update if different)
MODEL_PATH = '/content/drive/MyDrive/Colab_Data/thin_cloud_v2/thin_cloud_90000_steps.zip'

print(f"🔄 Loading model from: {MODEL_PATH}")

try:
    model = PPO.load(MODEL_PATH)
    print("\n✅ MODEL LOADED SUCCESSFULLY!")
    print("\n📋 Model Info:")
    print(f"   Policy: {model.policy.__class__.__name__}")
    print(f"   Learning rate: {model.learning_rate}")
    print(f"   N steps: {model.n_steps}")
    print(f"   Batch size: {model.batch_size}")
    print(f"   N epochs: {model.n_epochs}")
    print(f"   Gamma: {model.gamma}")
    
except Exception as e:
    print(f"\n❌ Error loading model: {e}")

In [ ]:
# Test inference with a dummy observation
import numpy as np

print("🧪 Testing model inference...\n")

# Create a dummy observation matching the environment's observation space
# ThinCloudDetectionEnv uses 7 features: [cnn_mean, cnn_std, thin_pct, edge_pct, current_thresh, bright_pct, ndsi_mean]
dummy_obs = np.array([0.5, 0.2, 0.3, 0.1, 0.4, 0.4, 0.2], dtype=np.float32)

try:
    action, _ = model.predict(dummy_obs, deterministic=True)
    print("✅ Model inference successful!")
    print(f"\n📊 Test input: {dummy_obs}")
    print(f"📊 Model output (action): {action}")
    print(f"   - Threshold delta: {action[0]:.4f}")
    print(f"   - Thin boost: {action[1]:.4f}")
except Exception as e:
    print(f"❌ Inference error: {e}")

## ✅ Summary

If all cells above ran successfully, your model is:
1. **Saved** on Google Drive (persistent storage)
2. **Loadable** with stable-baselines3
3. **Functional** and ready for inference

### Model Details
- **Location**: `/content/drive/MyDrive/Colab_Data/thin_cloud_v2/thin_cloud_90000_steps.zip`
- **Training**: 90,000 steps with multi-objective reward
- **Results**: +41.50% thin cloud recall improvement